In [0]:

import sys
PROJECT_ROOT = "/Workspace/Users/rohan.m.mukherjee@gmail.com/bfsi-lakehouse-databricks"
for p in [PROJECT_ROOT, f"{PROJECT_ROOT}/utils"]:
    if p not in sys.path:
        sys.path.insert(0, p)



In [0]:
%sql

-- SELECT * FROM bfsi_lakehouse.metadata.etl_process_master;
SELECT * FROM bfsi_lakehouse.metadata.etl_process_log;

In [0]:
%sql
TRUNCATE TABLE bfsi_lakehouse.metadata.etl_process_master;
TRUNCATE TABLE bfsi_lakehouse.metadata.etl_process_log;

In [0]:
%sql
-- SELECT * FROM bfsi_lakehouse.metadata.table_config;
SELECT * FROM bfsi_lakehouse.metadata.table_process_config;

-- Active Table Names
SELECT 
    t_cfg.table_id,
    t_cfg.source_table_name,
    t_process_cfg.process_type,
    t_process_cfg.load_type,
    t_process_cfg.is_active
FROM bfsi_lakehouse.metadata.table_config t_cfg
INNER JOIN bfsi_lakehouse.metadata.table_process_config t_process_cfg
ON t_cfg.table_id = t_process_cfg.table_id
WHERE t_process_cfg.is_active = TRUE
AND t_process_cfg.process_type = 'BRONZE';  -- cfg.ProcessType.BRONZE_INGEST

SELECT 
    t_process_cfg.write_strategy
FROM bfsi_lakehouse.metadata.table_config t_cfg
INNER JOIN bfsi_lakehouse.metadata.table_process_config t_process_cfg
ON t_cfg.table_id = t_process_cfg.table_id
WHERE t_process_cfg.is_active = TRUE
AND t_process_cfg.process_type = 'SILVER'
AND t_cfg.source_table_name = 't_Client'


In [0]:
%sql
UPDATE bfsi_lakehouse.metadata.table_config
SET
    source_path_pattern = REPLACE(
        source_path_pattern,
        'source_synthetic_data',
        'synthetic_data'
    ),
    last_edited_at = CURRENT_TIMESTAMP(),
    last_edited_by = 'rohan.m.mukherjee@gmail.com'
WHERE source_path_pattern LIKE '%source_synthetic_data%';

In [0]:
spark.read.parquet("/Volumes/bfsi_lakehouse/raw/synthetic_data/t_Client/dt=2024-01-02/").display(10)
# "/Volumes/bfsi_lakehouse/raw/synthetic_data/"


In [0]:
%sql
SELECT MAX(version)
FROM (DESCRIBE HISTORY bfsi_lakehouse.bronze.t_Client);

In [0]:
%sql
SELECT * FROM bfsi_lakehouse.metadata.etl_process_master;
SELECT * FROM bfsi_lakehouse.metadata.etl_process_log;

-- TRUNCATE TABLE bfsi_lakehouse.metadata.etl_process_master;
-- TRUNCATE TABLE bfsi_lakehouse.metadata.etl_process_log;

-- DROP TABLE bfsi_lakehouse.bronze.t_client
-- SELECT * FROM bfsi_lakehouse.bronze.t_client;
-- SELECT * FROM bfsi_lakehouse.bronze.t_AccountCustomer;

%sql

SELECT * FROM bfsi_lakehouse.metadata.etl_process_log WHERE run_id = 'c8c0866b-bfd4-46b9-9a3d-736eaeab54b6'



In [0]:
%sql
DROP TABLE bfsi_lakehouse.bronze.t_client;
DROP TABLE bfsi_lakehouse.bronze.t_AccountCustomer;
DROP TABLE bfsi_lakehouse.bronze.t_Loan;
DROP TABLE bfsi_lakehouse.bronze.t_LoanInstallment;
DROP TABLE bfsi_lakehouse.bronze.t_AccountTrx;




SELECT * FROM bfsi_lakehouse.bronze.t_client;
SELECT * FROM bfsi_lakehouse.bronze.t_AccountCustomer;

DESCRIBE bfsi_lakehouse.bronze.t_client;
DESCRIBE bfsi_lakehouse.metadata.input_column_config;

SELECT * FROM bfsi_lakehouse.metadata.input_column_config;



In [0]:
%sql
SELECT * FROM bfsi_lakehouse.metadata.schema_drift_log;
DESCRIBE bfsi_lakehouse.metadata.schema_drift_log;


In [0]:
%sql
INSERT INTO bfsi_lakehouse.metadata.input_column_config (
    -- column_id,
    table_id,
    process_type,
    source_column_name,
    target_column_name,
    data_type,
    is_nullable,
    is_pii,
    column_purpose,
    is_active,
    created_at,
    created_by,
    last_edited_at,
    last_edited_by
)
VALUES (
    6,
    'SILVER',
    'TestMissingColumn',
    NULL,
    'STRING',
    FALSE,
    FALSE,
    'ATTRIBUTE',
    TRUE,
    now(),
    'Rohan',
    now(),
    'Rohan'
);

In [0]:
%sql
SELECT * FROM bfsi_lakehouse.metadata.input_column_config WHERE table_id = 6;

UPDATE bfsi_lakehouse.metadata.input_column_config
SET is_active = FALSE
WHERE table_id = 6 AND column_id = 13;

In [0]:
%sql
SELECT * FROM bfsi_lakehouse.metadata.table_config;

SELECT 
    *
FROM bfsi_lakehouse.metadata.input_column_config
WHERE table_id = 11
AND process_type = 'SILVER'
AND is_active = true
ORDER BY column_id ASC

In [0]:
%sql
-- Step 1: Add column
ALTER TABLE bfsi_lakehouse.metadata.input_column_config
ADD COLUMN transform_expr STRING
COMMENT 'Spark SQL expression for DERIVED columns. NULL for normal columns. e.g. current_timestamp(), date_trunc("month", DueDate)';


-- Step 2: Update existing _silver_processed_at row
UPDATE bfsi_lakehouse.metadata.input_column_config
SET transform_expr = 'current_timestamp()'
WHERE source_column_name IS NULL
  AND target_column_name = '_silver_processed_at';


-- Step 3: Verify
SELECT *
FROM bfsi_lakehouse.metadata.input_column_config
WHERE is_nullable = False;

In [0]:
%sql
SELECT 
        table_id,
        source_table_name,
        source_system,
        source_format,
        source_path_pattern,
        target_catalog,
        target_schema,
        target_table,
        partition_cols,
        replace_where_template,
        drift_policy,
        load_priority,
        is_active
    FROM bfsi_lakehouse.metadata.table_config
    WHERE source_table_name = 't_Client' AND process_type = 'SILVER'


SELECT * FROM bfsi_lakehouse.metadata.schema_drift_log

SELECT * FROM bfsi_lakehouse.bronze.t_Client;
SELECT * FROM bfsi_lakehouse.bronze.t_AccountCustomer;

